In [1]:
import torch
import torch.nn as nn
import pandas as pd
import optuna
import torchmetrics

torch.manual_seed(42)
device = "cuda"

In [2]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

cifar10_mean = (0.4914, 0.4822, 0.4465)
cifar10_std  = (0.2470, 0.2435, 0.2616)

transform = transforms.Compose([
    transforms.ToTensor(),                               # PIL Image → FloatTensor [0,1]
    transforms.Normalize(mean=cifar10_mean, std=cifar10_std)
])

train_dataset = datasets.CIFAR10(root="datasets/", train=True,  download=True, transform=transform)
test_dataset  = datasets.CIFAR10(root="datasets/", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_dataset,  batch_size=32, shuffle=False)

images, labels = next(iter(train_loader))
print(images.shape)  # (32 images)

torch.Size([32, 3, 32, 32])


In [13]:
from typing import Any

class Dense(nn.Module):
    def __init__(self, input_n:int, output_n:int):
        super().__init__()
        self.linear = nn.Linear(input_n, output_n)
        self.activation = nn.SiLU()
        
        nn.init.kaiming_normal_(self.linear.weight, nonlinearity="relu")
        nn.init.zeros_(self.linear.bias)
        
    def forward(self, x):
        return self.activation(self.linear(x))

hidden_layers = [Dense(100, 100) for _ in range(20)]

model = nn.Sequential(
    nn.Flatten(),
    Dense(3*32*32, 100),
    *hidden_layers,
    nn.Linear(100,10)
)
model.to(device)

Sequential(
  (0): Flatten(start_dim=1, end_dim=-1)
  (1): Dense(
    (linear): Linear(in_features=3072, out_features=100, bias=True)
    (activation): SiLU()
  )
  (2): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (3): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (4): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (5): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (6): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (7): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (8): Dense(
    (linear): Linear(in_features=100, out_features=100, bias=True)
    (activation): SiLU()
  )
  (9): Dense(
    (linear): Linear(in_features=100, out_features=10

In [4]:
print(f"Train samples : {len(train_dataset)}")   # 50,000
print(f"Test  samples : {len(test_dataset)}")    # 10,000

image, label = train_dataset[0]
print(f"Image shape   : {image.shape}")          # torch.Size([3, 32, 32])
print(f"Label         : {label}")                # int 0~9


Train samples : 50000
Test  samples : 10000
Image shape   : torch.Size([3, 32, 32])
Label         : 6


In [5]:
torch.unique(torch.tensor(test_dataset.targets))

tensor([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])

In [14]:
import torchmetrics
from torch.optim import Optimizer
from torch.utils.tensorboard import SummaryWriter

#tensorboard --logdir=study\c_11\log

def train(writer:SummaryWriter, model:nn.Module, optimizer:Optimizer, criterion, accuracy:torchmetrics.Accuracy, 
          train_loader:DataLoader, test_loader:DataLoader, n_epoch:int = 1):
    model.train()
    global_step = 0
    
    for epoch_counter in range(n_epoch):
        total_loss:int = 0
        for X, y in train_loader:
            X,y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = criterion(y_pred, y)
            loss.backward()
            optimizer.step()
            optimizer.zero_grad()
            total_loss += loss.item()
            
            writer.add_scalar("Loss_batch/train", loss.item(), global_step)
            global_step += 1
        
        
        avg_loss = total_loss / len(train_loader)
        avg_loss_test, acc = eval(model, criterion, accuracy, test_loader)
        
        print(f"epoch:{epoch_counter}, avg_loss:{avg_loss}, acc={acc}")
        
        writer.add_scalars('Loss', {
            'train': avg_loss,
            'test': avg_loss_test
        }, epoch_counter)
        writer.add_scalar("acc/test", acc, epoch_counter)
        writer.flush()

def eval(model:nn.Module, criterion, metric_fn:torchmetrics.Accuracy, 
         dataloader:DataLoader, agg_fn = torch.mean) -> tuple[float, float]:
    model.eval()
    metrics = []
    total_loss = 0
    with torch.no_grad():
        for X, y in dataloader:
            X,y = X.to(device), y.to(device)
            y_pred = model(X)
            loss = criterion(y_pred, y)
            total_loss += loss.item()
            metric = metric_fn(y_pred, y)
            metrics.append(metric)

    avg_loss = total_loss/len(dataloader)
    metric_fn.reset()
    model.train()
    return avg_loss, agg_fn(torch.stack(metrics))

In [12]:
n_epoch:int = 15
lr = 1e-3
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-2, betas=(0.9, 0.999), eps=1e-8)
accuracy = torchmetrics.Accuracy(task="multiclass", num_classes=10).to(device)

with SummaryWriter("study/c_11/log/normal") as writer:
    train(writer, model, optimizer, criterion, accuracy, train_loader, test_loader, n_epoch=n_epoch)

epoch:0, avg_loss:1.290984250114121, acc=0.48512378334999084
epoch:1, avg_loss:1.2685740436412398, acc=0.4894169270992279
epoch:2, avg_loss:1.2579199761362008, acc=0.4945087730884552
epoch:3, avg_loss:1.2215513115270886, acc=0.48961660265922546
epoch:4, avg_loss:1.2033106686400818, acc=0.49241212010383606
epoch:5, avg_loss:1.1823044136328644, acc=0.49880191683769226
epoch:6, avg_loss:1.1678535052010897, acc=0.500898540019989
epoch:7, avg_loss:1.1443412523199485, acc=0.5006988644599915
epoch:8, avg_loss:1.129486819947292, acc=0.5049920082092285
epoch:9, avg_loss:1.1224072206424585, acc=0.4939097464084625
epoch:10, avg_loss:1.1132293492269607, acc=0.4962060749530792
epoch:11, avg_loss:1.0792497980312439, acc=0.503494381904602
epoch:12, avg_loss:1.0725338258998027, acc=0.5001996755599976
epoch:13, avg_loss:1.0627414448018724, acc=0.5081868767738342
epoch:14, avg_loss:1.0487795359266163, acc=0.5015974640846252
